#### IMPORTING NECESSARY LIBRARIES


In [50]:
import numpy as np
import pandas as pd

In [51]:
# Extracting data
yankee_df = pd.read_csv(r'C:\Users\user\Desktop\Yankee Ecommerce\Yankee_Ecommerce_Study_Case\dataset\raw data\yanki_ecommerce.csv')

In [52]:
yankee_df.columns


Index(['Order_ID', 'Customer_ID', 'Customer_Name', 'Product_ID',
       'Product_Name', 'Brand', 'Category', 'Price', 'Quantity', 'Total_Price',
       'Order_Date', 'Shipping_Address', 'City', 'State', 'Country',
       'Postal_Code', 'Email', 'Phone_Number', 'Payment_Method',
       'Transaction_Status'],
      dtype='object')

### DATA CLEANING

In [53]:


#drop missing values
yankee_df.dropna(subset =['Order_ID' , 'Customer_ID'], inplace=True)

In [54]:
yankee_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1000 entries, 2 to 1019
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Order_ID            1000 non-null   object 
 1   Customer_ID         1000 non-null   object 
 2   Customer_Name       1000 non-null   object 
 3   Product_ID          1000 non-null   object 
 4   Product_Name        1000 non-null   object 
 5   Brand               1000 non-null   object 
 6   Category            1000 non-null   object 
 7   Price               1000 non-null   float64
 8   Quantity            1000 non-null   int64  
 9   Total_Price         1000 non-null   float64
 10  Order_Date          1000 non-null   object 
 11  Shipping_Address    1000 non-null   object 
 12  City                1000 non-null   object 
 13  State               999 non-null    object 
 14  Country             1000 non-null   object 
 15  Postal_Code         1000 non-null   int64  
 16  Email      

In [55]:
# convert Order_Date from string to datetime
yankee_df['Order_Date'] = pd.to_datetime(
    yankee_df['Order_Date'],
    dayfirst=True
)

In [56]:
# customer table
customer_df  = yankee_df[[ 'Customer_ID', 'Customer_Name' , 'Email', 'Phone_Number']].copy().drop_duplicates().reset_index(drop= True)
customer_df.head()

,Customer_ID,Customer_Name,Email,Phone_Number
0,e0d6cb3c-c4b0-4cfe-8225-b65d094d2424,Dominic Buchanan,margaret97@example.com,259.603.6134
1,fa3ca35a-5540-404b-a7eb-9001cdcbd840,Daniel Allen,angela55@example.com,+1-869-659-4272x982
2,7ad4de53-e6d7-4cd3-99b8-13fb70fe7a34,Daniel Schmidt,wayne59@example.org,292.840.0975x724
3,4b9b409c-19f2-41c0-bc7c-6556e0647ebb,John Gonzalez,qsherman@example.com,+1-643-561-3912x262
4,81b513ad-5c02-48cf-bff0-39e1440c4d22,Amber Benitez,derek85@example.com,3424148376


In [57]:
#products table

products_df = yankee_df[['Product_ID',
       'Product_Name', 'Brand', 'Category', 'Price',]].copy().drop_duplicates().reset_index(drop= True)
products_df.head()

,Product_ID,Product_Name,Brand,Category,Price
0,2ef6e8fa-6a36-4515-b1c2-a0a700abf386,despite,"Lawson, Stone and Campos",perfume oil,250.57
1,3ba38e01-f8e7-4af2-9246-87ef0961d4f5,sea,Washington Group,perfume oil,179.81
2,a58c53bd-a34b-4541-b926-bec9eb84cac2,suddenly,Rodgers Ltd,perfume oil,600.55
3,e6021be5-90be-4199-b6c6-82542fb2973c,site,"Wilson, Scott and Johnson",perfume oil,414.36
4,60127671-ab32-4fb5-bfeb-1a7782ad835e,act,"Riddle, Alvarez and Robinson",perfume,704.05


In [58]:
# shipping table
shipping_address_df = yankee_df[['Customer_ID','Shipping_Address', 'City', 'State', 'Country',
       'Postal_Code']].copy().drop_duplicates().reset_index(drop= True) 

shipping_address_df.index.name = 'shipping_ID'
shipping_address_df =shipping_address_df.reset_index()


In [59]:
# orders table
orders_df = yankee_df[['Order_ID', 'Customer_ID','Product_ID','Quantity', 'Total_Price' , 'Order_Date']].copy().drop_duplicates().reset_index(drop= True)




In [60]:
#payment methods table

payment_method_df = yankee_df[['Order_ID', 'Payment_Method',
       'Transaction_Status']]

In [61]:
customer_df.to_csv(r'dataset\clean data\customers.csv' , index = False)
products_df.to_csv(r'dataset\clean data\products.csv' , index = False)
orders_df.to_csv(r'dataset\clean data\orders.csv' , index = False)
shipping_address_df.to_csv(r'dataset\clean data\shipping_address.csv' , index = False)
payment_method_df.to_csv(r'dataset\clean data\payment_method.csv' , index = False)


In [62]:
yankee_df.columns

Index(['Order_ID', 'Customer_ID', 'Customer_Name', 'Product_ID',
       'Product_Name', 'Brand', 'Category', 'Price', 'Quantity', 'Total_Price',
       'Order_Date', 'Shipping_Address', 'City', 'State', 'Country',
       'Postal_Code', 'Email', 'Phone_Number', 'Payment_Method',
       'Transaction_Status'],
      dtype='object')

### Data Loading 


In [63]:
pip install psycopg2

Note: you may need to restart the kernel to use updated packages.


In [65]:
import psycopg2

In [66]:
def get_db_connection():
  connection = psycopg2.connect(
     host = 'localhost' ,
     database = 'Yanki Ecommerce',
     user = 'postgres' ,
     password = 'Nurudeen03@'
  )
  return connection

In [67]:
conn = get_db_connection()

In [68]:
def create_table():
    conn = get_db_connection()
    cursor = conn.cursor()

    create_table_query = '''
        CREATE SCHEMA IF NOT EXISTS yanki;

        DROP TABLE IF EXISTS yanki.payment_method CASCADE;
        DROP TABLE IF EXISTS yanki.orders CASCADE;
        DROP TABLE IF EXISTS yanki.shipping_address CASCADE;
        DROP TABLE IF EXISTS yanki.products CASCADE;
        DROP TABLE IF EXISTS yanki.customer CASCADE;

        CREATE TABLE IF NOT EXISTS yanki.customer(
            Customer_ID UUID PRIMARY KEY,
            Customer_Name TEXT,
            Email TEXT,
            Phone_Number TEXT
        );

        CREATE TABLE IF NOT EXISTS yanki.products(
            Product_ID UUID PRIMARY KEY,
            Product_Name TEXT,
            Brand TEXT,
            Category TEXT,
            Price FLOAT
        );

        CREATE TABLE IF NOT EXISTS yanki.shipping_address(
            Shipping_ID SERIAL PRIMARY KEY,
            Customer_ID UUID,
            Shipping_Address TEXT,
            City TEXT,
            State TEXT,
            Country TEXT,
            Postal_Code INTEGER,
            FOREIGN KEY (Customer_ID)
                REFERENCES yanki.customer(Customer_ID)
        );

        CREATE TABLE IF NOT EXISTS yanki.orders(
            Order_ID UUID PRIMARY KEY,
            Customer_ID UUID,
            Product_ID UUID,
            Quantity INTEGER,
            Total_Price FLOAT,
            Order_Date DATE,
            FOREIGN KEY (Customer_ID)
                REFERENCES yanki.customer(Customer_ID),
            FOREIGN KEY (Product_ID)
                REFERENCES yanki.products(Product_ID)
        );

        CREATE TABLE IF NOT EXISTS yanki.payment_method(
            Order_ID UUID,
            Payment_Method TEXT,
            Transaction_Status TEXT,
            FOREIGN KEY (Order_ID)
                REFERENCES yanki.orders(Order_ID)
        );
    '''

    cursor.execute(create_table_query)
    conn.commit()
    cursor.close()
    conn.close()

In [69]:
create_table()

In [70]:
display(orders_df.columns)
display(orders_df.info())

Index(['Order_ID', 'Customer_ID', 'Product_ID', 'Quantity', 'Total_Price',
       'Order_Date'],
      dtype='object')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 990 entries, 0 to 989
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Order_ID     990 non-null    object        
 1   Customer_ID  990 non-null    object        
 2   Product_ID   990 non-null    object        
 3   Quantity     990 non-null    int64         
 4   Total_Price  990 non-null    float64       
 5   Order_Date   990 non-null    datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(1), object(3)
memory usage: 46.5+ KB


None

In [72]:
import csv

def load_data_from_csv(csv_path):

    conn = get_db_connection()
    cursor = conn.cursor()

    with open(csv_path, 'r') as file:

        reader = csv.reader(file)

        next(reader)

        for row in reader:

            cursor.execute('''
                INSERT INTO yanki.customer
                (Customer_ID, Customer_Name, Email, Phone_Number)

                VALUES(%s, %s, %s, %s);
            ''', row)

    conn.commit()
    cursor.close()
    conn.close()


csv_file_path = r'dataset\clean data\customers.csv'

load_data_from_csv(csv_file_path)

In [74]:
import csv

def load_data_from_csv(csv_path):

    conn = get_db_connection()
    cursor = conn.cursor()

    with open(csv_path, 'r') as file:

        reader = csv.reader(file)

        next(reader)

        for row in reader:

            cursor.execute('''
                INSERT INTO yanki.products
                (Product_ID, Product_Name, Brand, Category , Price)

                VALUES(%s, %s, %s, %s , %s);
            ''', row)

    conn.commit()
    cursor.close()
    conn.close()


csv_file_path = r'dataset\clean data\products.csv'

load_data_from_csv(csv_file_path)

In [78]:
import csv

def load_data_from_csv(csv_path):

    conn = get_db_connection()
    cursor = conn.cursor()

    with open(csv_path, 'r') as file:

        reader = csv.reader(file)

        next(reader)

        for row in reader:

            cursor.execute('''
                INSERT INTO yanki.shipping_address
                (Shipping_ID, Customer_ID ,Shipping_Address , City, State ,Country , Postal_Code)

                VALUES(%s,%s,%s,%s,%s,%s,%s);
            ''', row)

    conn.commit()
    cursor.close()
    conn.close()


csv_file_path = r'dataset\clean data\shipping_address.csv'

load_data_from_csv(csv_file_path)

In [79]:
import csv

def load_data_from_csv(csv_path):

    conn = get_db_connection()
    cursor = conn.cursor()

    with open(csv_path, 'r') as file:

        reader = csv.reader(file)

        next(reader)

        for row in reader:

            cursor.execute('''
                INSERT INTO yanki.orders
                (Order_ID, Customer_ID ,Product_ID , Quantity, Total_Price ,Order_Date )

                VALUES(%s,%s,%s,%s,%s,%s);
            ''', row)

    conn.commit()
    cursor.close()
    conn.close()


csv_file_path = r'dataset\clean data\orders.csv'

load_data_from_csv(csv_file_path)

In [80]:
import csv

def load_data_from_csv(csv_path):

    conn = get_db_connection()
    cursor = conn.cursor()

    with open(csv_path, 'r') as file:

        reader = csv.reader(file)

        next(reader)

        for row in reader:

            cursor.execute('''
                INSERT INTO yanki.payment_method
                (Order_ID,Payment_Method ,Transaction_Status)

                VALUES(%s,%s,%s);
            ''', row)

    conn.commit()
    cursor.close()
    conn.close()


csv_file_path = r'dataset\clean data\payment_method.csv'

load_data_from_csv(csv_file_path)
